## TCGA-LUAD
* 研究路线：数据收集→预处理→核心多组学分析→分子分型→药敏整合→临床验证全流程
* 参考文献：1

In [1]:
### 环境配置

In [2]:
set.seed(42)

target_dir = "C:\\Users\\wenfang\\Desktop\\TCGA-LUAD\\"
result_dir = paste(target_dir, "result\\preprocess\\", sep="")
if (!file.exists(result_dir)){
    dir.create(result_dir, recursive = TRUE)
}
print(result_dir)

[1] "C:\\Users\\wenfang\\Desktop\\TCGA-LUAD\\result\\preprocess\\"


### 数据收集
* TCGA-LUAD: HM450甲基化数据beta, 转录组RSEM基因表达数据，CNV数据，mc3突变数据....(UCSC Xena)

### 样本对齐
* 仅保留原发肿瘤样本（正常对照仅用于差异分析）

In [3]:
# 读取表型数据
phenotypes <- read.delim(paste(target_dir, "Phenotypes-TCGA.LUAD.sampleMap_LUAD_clinicalMatrix", sep=""), row.names=1, header=TRUE, sep="\t", check.names=FALSE)

# 筛选保留原发肿瘤样本和正常对照
phenotypes_filtered <- phenotypes[phenotypes$sample_type %in% c("Primary Tumor", "Solid Tissue Normal"),]
head(phenotypes_filtered[,1:5])

# Primary Tumor & Solid Tissue Normal
tumor_filtered <- phenotypes[phenotypes$sample_type == "Primary Tumor",]
normal_filtered <- phenotypes[phenotypes$sample_type == "Solid Tissue Normal",]
#print(dim(tumor_filtered)) # Tumor samples: 520, 
#print(dim(normal_filtered)) # "Normal samples: 120

# 提取待分析样本ID
target_sample_ids <- rownames(phenotypes_filtered)
#print(length(target_sample_ids)) # 640

,ABSOLUTE_Ploidy,ABSOLUTE_Purity,AKT1,ALK_translocation,BRAF
,<dbl>,<dbl>,<chr>,<chr>,<chr>
TCGA-05-4244-01,NA,NA,,,
TCGA-05-4249-01,3.77,0.46,none,,p.A762E
TCGA-05-4250-01,NA,NA,,,
TCGA-05-4382-01,NA,NA,none,,p.L613F
TCGA-05-4384-01,2.04,0.48,none,,none
TCGA-05-4389-01,3.29,0.48,none,,p.G469V


#### 甲基化数据————去除低质量探针/样本，消除性别干扰，确保数据可靠性

In [4]:
# 读取甲基化beta矩阵
meth_beta <- read.delim(paste(target_dir, "TCGA.LUAD.sampleMap_HumanMethylation450\\HumanMethylation450", sep=""), row.names=1, header=TRUE, sep="\t", check.names=FALSE)

# 提取肿瘤样本
meth_beta_tumor <- meth_beta[,intersect(colnames(meth_beta), rownames(tumor_filtered))]

* 质控与过滤

In [4]:
library(IlluminaHumanMethylation450kanno.ilmn12.hg19)

# 保留CpG位点
cg_probes <- rownames(meth_beta_tumor)[grepl("^cg", rownames(meth_beta_tumor))]
meth_beta_tumor_cg <- meth_beta_tumor[cg_probes,]

# 探针过滤：保留在≥90% 样本中检测到的探针 (探针缺失率<=10%)
meth_beta_cg_filtered <- meth_beta_tumor_cg[rowMeans(is.na(meth_beta_tumor_cg)) <= 0.1,]
print(dim(meth_beta_cg_filtered))

# 样本过滤：保留在>=95%探针中有效的样本 （样本缺失率<5%）
meth_beta_cg_sample_filtered <- meth_beta_cg_filtered[, colMeans(is.na(meth_beta_cg_filtered)) <= 0.2]
print(dim(meth_beta_cg_sample_filtered))

# 去除在chrX, chrY的探针
cg_chrom <- read.delim(paste(target_dir, "probeMap_illuminaMethyl450_hg19_GPL16304_TCGAlegacy", sep=""), row.names=1, header=TRUE, sep="\t", check.names=FALSE)
cg_chrXY_filtered <- cg_chrom[!cg_chrom$chrom %in% c("chrX", "chrY"),]
common_cg_ids <- intersect(rownames(meth_beta_cg_sample_filtered), rownames(cg_chrXY_filtered))
meth_beta_cg_sample_chrXY_filtered <- meth_beta_cg_sample_filtered[common_cg_ids,]
cat("探针数量：", nrow(meth_beta_cg_sample_chrXY_filtered), "\n")

# 去除覆盖SNP的探针————Probe_rs ≠ NA → 探针本身覆盖 SNP；CpG_rs ≠ NA → CpG 位点覆盖 SNP；SBE_rs ≠ NA → 探针侧翼覆盖 SNP
probe_anno <- getAnnotation(IlluminaHumanMethylation450kanno.ilmn12.hg19)
anno <- probe_anno[rownames(meth_beta_cg_sample_chrXY_filtered), ]

snp_probes <- rownames(anno)[!is.na(anno$Probe_rs) | !is.na(anno$CpG_rs) | !is.na(anno$SBE_rs)]
meth_snp_filter <- meth_beta_cg_sample_chrXY_filtered[!rownames(meth_beta_cg_sample_chrXY_filtered) %in% snp_probes, ]
cat("去除SNP覆盖探针数量：", length(snp_probes), "\n") 
cat("质控后剩余探针数量：", nrow(meth_snp_filter), "\n")
head(meth_snp_filter[1:5,1:5])

Loading required package: minfi

Warning message:
"package 'minfi' was built under R version 4.5.2"
Loading required package: BiocGenerics

Warning message:
"package 'BiocGenerics' was built under R version 4.5.2"
Loading required package: generics

Warning message:
"package 'generics' was built under R version 4.5.2"

Attaching package: 'generics'


The following objects are masked from 'package:base':

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: 'BiocGenerics'


The following objects are masked from 'package:stats':

    IQR, mad, sd, var, xtabs


The following objects are masked from 'package:base':

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, t

[1] 393539    457
[1] 393539    457
探针数量： 383765 
去除SNP覆盖探针数量： 64685 
质控后剩余探针数量： 319080 


,TCGA-44-4112-01,TCGA-NJ-A4YP-01,TCGA-86-8278-01,TCGA-62-A470-01,TCGA-44-6778-01
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
cg13332474,0.06685,0.3178,0.4128,0.0397,0.0315
cg00651829,0.09115,0.2879,0.4702,0.0616,0.0424
cg17027195,0.03115,0.1224,0.0446,0.0292,0.0365
cg09868354,0.05725,0.1316,0.1409,0.0776,0.0600
cg03050183,0.15125,0.1107,0.1082,0.1142,0.0893


In [5]:
# release resource
rm(meth_beta, meth_beta_tumor, meth_beta_tumor_cg, meth_beta_cg_filtered, meth_beta_cg_sample_filtered, cg_chrom, cg_chrXY_filtered, common_cg_ids)
rm(meth_beta_cg_sample_chrXY_filtered, probe_anno)

gc()

Warning message in rm(meth_beta, meth_beta_tumor, meth_beta_tumor_cg, meth_beta_cg_filtered, :
"object 'meth_beta_tumor_cg' not found"
Warning message in rm(meth_beta, meth_beta_tumor, meth_beta_tumor_cg, meth_beta_cg_filtered, :
"object 'meth_beta_cg_filtered' not found"
Warning message in rm(meth_beta, meth_beta_tumor, meth_beta_tumor_cg, meth_beta_cg_filtered, :
"object 'meth_beta_cg_sample_filtered' not found"
Warning message in rm(meth_beta, meth_beta_tumor, meth_beta_tumor_cg, meth_beta_cg_filtered, :
"object 'cg_chrom' not found"
Warning message in rm(meth_beta, meth_beta_tumor, meth_beta_tumor_cg, meth_beta_cg_filtered, :
"object 'cg_chrXY_filtered' not found"
Warning message in rm(meth_beta, meth_beta_tumor, meth_beta_tumor_cg, meth_beta_cg_filtered, :
"object 'common_cg_ids' not found"
Warning message in rm(meth_beta_cg_sample_chrXY_filtered, probe_anno):
"object 'meth_beta_cg_sample_chrXY_filtered' not found"
Warning message in rm(meth_beta_cg_sample_chrXY_filtered, probe_an

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,675016,36.1,1877888,100.3,1453849,77.7
Vcells,1735162,13.3,293267564,2237.5,366478548,2796.1


* 缺失值填充

In [6]:
#library(impute)
library(matrixStats) # 加速
set.seed(42)

# KNN 
#meth_beta_imputed <- impute.knn(as.matrix(meth_beta_sample_cg_chrXYM_filtered), k=10, rng.seed=42)$data # 近邻数选10（样本多的情况下，k可适当增大）

# 中位数填充
dat <- as.matrix(meth_snp_filter)
med <- rowMedians(dat, na.rm = TRUE)
meth_beta_imputed <- dat
meth_beta_imputed[is.na(meth_beta_imputed)] <- med[row(meth_beta_imputed)[is.na(meth_beta_imputed)]]

# 截断beta值到0~1（避免插补值超出合理范围）
meth_beta_imputed[meth_beta_imputed < 0] <- 0
meth_beta_imputed[meth_beta_imputed > 1] <- 1
head(meth_beta_imputed[,1:5])

,TCGA-44-4112-01,TCGA-NJ-A4YP-01,TCGA-86-8278-01,TCGA-62-A470-01,TCGA-44-6778-01
cg13332474,0.06685,0.3178,0.4128,0.0397,0.0315
cg00651829,0.09115,0.2879,0.4702,0.0616,0.0424
cg17027195,0.03115,0.1224,0.0446,0.0292,0.0365
cg09868354,0.05725,0.1316,0.1409,0.0776,0.0600
cg03050183,0.15125,0.1107,0.1082,0.1142,0.0893
cg04244855,0.84520,0.8653,0.8593,0.6005,0.8478


* 探针-基因映射

In [7]:
library(dplyr)
library(tibble)

# 探针-基因映射————probes corresponding to multiple gene names were excluded, while an average value was calculated when multiple probes mapped to one gene. 
#head(colnames(anno[1:5,]))
filter_probes <- rownames(anno)[!is.na(anno$UCSC_RefGene_Name) & anno$UCSC_RefGene_Name != "" & !grepl(";", anno$UCSC_RefGene_Name)]
common_probes <- intersect(filter_probes, rownames(meth_beta_imputed))
cat("有效探针数：",length(common_probes))
    
meth_filter <- meth_beta_imputed[common_probes, ]
meth_anno <- anno[common_probes, ]

meth_filter_df <- meth_filter %>% 
  as.data.frame() %>% 
  rownames_to_column(var = "probe")

meth_anno_df <- meth_anno %>% 
  as.data.frame() %>%
  rownames_to_column(var = "probe") %>%
  select(probe, UCSC_RefGene_Name)

df_merge <- inner_join(meth_filter_df, meth_anno_df, by = "probe")
meth_gene <- df_merge %>% 
  group_by(UCSC_RefGene_Name) %>%
  summarise(across(where(is.numeric), ~ mean(.x, na.rm = TRUE))) %>%  # 仅数值列（样本）取均值
  ungroup()

gene_names <- meth_gene[[1]]
meth_gene_matrix <- as.matrix(meth_gene[, -1])
rownames(meth_gene_matrix) <- gene_names  

head(meth_gene_matrix[1:5, 1:5])

cat("甲基化探针映射-基因数：", nrow(meth_gene_matrix), " | 样本数：", ncol(meth_gene_matrix), "\n")

Warning message:
"package 'dplyr' was built under R version 4.5.2"

Attaching package: 'dplyr'


The following object is masked from 'package:minfi':

    combine


The following objects are masked from 'package:Biostrings':

    collapse, intersect, setdiff, setequal, union


The following object is masked from 'package:XVector':

    slice


The following object is masked from 'package:Biobase':

    combine


The following object is masked from 'package:matrixStats':

    count


The following objects are masked from 'package:GenomicRanges':

    intersect, setdiff, union


The following object is masked from 'package:Seqinfo':

    intersect


The following objects are masked from 'package:IRanges':

    collapse, desc, intersect, setdiff, slice, union


The following objects are masked from 'package:S4Vectors':

    first, intersect, rename, setdiff, setequal, union


The following objects are masked from 'package:BiocGenerics':

    combine, intersect, setdiff, setequal, union




有效探针数： 139992

,TCGA-44-4112-01,TCGA-NJ-A4YP-01,TCGA-86-8278-01,TCGA-62-A470-01,TCGA-44-6778-01
A1BG,0.3365000,0.4884000,0.4029000,0.2448500,0.2740000
A2LD1,0.7919111,0.8005111,0.7189111,0.6558889,0.7728333
A2M,0.7042188,0.7559500,0.7402625,0.3124750,0.5349750
A2ML1,0.8287100,0.7730200,0.7932800,0.4835200,0.6012000
A4GALT,0.4912719,0.4932437,0.4680313,0.4559500,0.4146063


甲基化探针映射-基因数： 13847  | 样本数： 457 


In [1]:
print(head(df_merge))

ERROR: Error: object 'df_merge' not found


In [8]:
# 保存数据
write.table(meth_snp_filter, paste0(result_dir,"meth_snp_filter_tumor.txt"), sep="\t")
write.table(meth_beta_imputed, paste0(result_dir,"meth_beta_imputed_tumor.txt"), sep="\t")
write.table(meth_gene_matrix, paste0(result_dir,"meth_gene_matrix_tumor.txt"), sep="\t")

In [ ]:
# 当要研究甲基化对基因表达的调控机制时，可考虑聚焦启动子/增强子
# 甲基化功能区域筛选————优先保留 “增强子 / 启动子区域” 探针（这些区域甲基化变化直接影响基因表达，预后关联更强）
#library(IlluminaHumanMethylation450kanno.ilmn12.hg19)

# 加载450K芯片官方注释
#probe_anno <- getAnnotation(IlluminaHumanMethylation450kanno.ilmn12.hg19)
#gene_groups <- as.character(unlist(probe_anno$UCSC_RefGene_Group))

# 保留功能区域探针并提取
#keep_regions <- c("TSS1500", "TSS200", "5'UTR", "Enhancer")

#target_region <- function(group_str) {
#    if(is.na(group_str)) return (FALSE)
#    regions <- strsplit(group_str, ";")[[1]]
#    any(regions %in% keep_regions)
#}

#keep_mask <- sapply(gene_groups, target_region)

#keep_probes <- rownames(probe_anno)[keep_mask]
#meth_filter <- meth_beta_sample_cg_chrXYM_filtered[rownames(meth_beta_sample_cg_chrXYM_filtered) %in% keep_probes, ]
#cat("甲基化功能区筛选完成\n保留全部探针数（启动子+增强子）：", nrow(meth_filter), "\n")
#head(meth_filter[,1:5])

In [9]:
rm(dat, med, filter_probes, common_probes, meth_filter, meth_anno, meth_filter_df, meth_anno_df, df_merge, meth_gene, gene_names)
rm(meth_snp_filter, meth_beta_imputed, meth_gene_matrix)

gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,13942248,744.6,43600482,2328.6,43600482,2328.6
Vcells,73088206,557.7,933739740,7123.9,1215712099,9275.2


#### 转录组数据
* This dataset shows the gene-level transcription estimates, as in log2(x+1) transformed RSEM normalized count.

In [10]:
library(dplyr)

# 读取count矩阵
rna_exp <- read.delim(paste(target_dir, "gene_expression_RNAseq-illumina-HiSeqV2", sep=""), row.names=1, header=TRUE, sep="\t", check.names=FALSE)
# dim(rna_exp) 20530 576

# 提取肿瘤样本
rna_exp_tumor <- rna_exp[,intersect(colnames(rna_exp), rownames(tumor_filtered))]

# 过滤低表达基因(保留至少10%的样本中表达量>1的基因)
keep_genes <- rowSums(rna_exp_tumor > 1) >= 0.1 * ncol(rna_exp_tumor)
rna_exp_filtered <- rna_exp_tumor[keep_genes, ]
cat("Raw genes:",nrow(rna_exp_tumor),"| After filtered:", nrow(rna_exp_filtered), "\n")
head(rna_exp_filtered[,1:5])

Raw genes: 20530 | After filtered: 18423 


,TCGA-69-7978-01,TCGA-62-8399-01,TCGA-78-7539-01,TCGA-73-4658-01,TCGA-44-6775-01
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
ARHGEF10L,9.9898,10.4257,9.6264,9.2078,10.0039
HIF3A,4.2598,11.6239,9.1362,5.0288,4.0573
RNF17,0.4181,0.0000,1.1231,0.0000,1.0069
RNF10,10.3657,11.5489,11.6692,11.6209,11.1721
RNF11,11.1718,11.0200,10.4679,11.3414,11.0969
RNF13,10.5897,9.2843,10.4649,10.9376,10.9337


In [11]:
write.table(rna_exp_filtered, paste0(result_dir,"rna_exp_filtered_tumor.txt"), sep="\t")

In [12]:
rm(rna_exp_tumor, keep_genes, rna_exp_filtered)

gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,13949553,745.0,43600482,2328.6,43600482,2328.6
Vcells,84943954,648.1,746991792,5699.1,1215712099,9275.2
